# T17 — Debugging a plate model: negative divergence at ridges, negative convergence at trenches, and transform velocities

**Cluster C: Plate-model debugging.** (1 / 4).

## What this notebook does

Plate models encode a huge amount of geometry: mid-ocean ridges (MORs), subduction zones (SZs), and transforms are drawn as line features and expected to move in specific ways when the rotation model is applied. In a well-constructed model, MOR sub-segments should DIVERGE, SZ sub-segments should CONVERGE, and transform sub-segments should have negligible orthogonal (across-boundary) velocity. When any of those expectations is violated for a sub-segment, it flags a **construction anomaly** — either the topology is wrong (e.g. a line drawn as a ridge but functioning as a trench), or the rotations aren't consistent with the topology, or a plate-ID is missing.

This notebook walks through the diagnostic pattern used to find those anomalies:

1. **§1** Load the plate model (Zahirovic 2022 by default) and the diagnostic helper `diagnose_topology_convergence` from the bundled `plate_model_debug` package.
2. **§2** For each snapshot in **100, 125, 150, 175, 200 Ma**, compute the orthogonal convergence velocity at every MOR / SZ / transform sub-segment midpoint.
3. **§3** Render one pyGMT map per snapshot highlighting:
   - MOR sub-segments with **positive** orthogonal convergence velocity (i.e. **negative divergence**, they should be spreading but aren't).
   - SZ sub-segments with **negative** orthogonal convergence velocity (i.e. **negative convergence**, they should be closing but aren't).
   - Transform sub-segments with any non-zero orthogonal convergence velocity (transforms should be strike-slip = orthogonal component ~ 0).
4. **§4** Build a **compact MP4 video** at 1-Myr cadence over the full 100-200 Ma window so the user can watch anomalies appear, migrate, and disappear as topologies change through time.
5. **§5** Extend this — swap in a different plate model, extend the time window, run over 0-1000 Ma, adjust the velocity thresholds.

**Reading the anomalies.** No plate model is perfect — every published model carries some residual number of these flags because fixing them all is time-consuming manual work. The point of the diagnostic is NOT that a clean model has zero anomalies (usually impossible) but that you can quickly SEE where the anomalies are, decide which matter for your downstream analysis, and either fix them or work around them.

**How to interpret the results — the maps are a triage tool, not an authoritative error catalogue.** Every flagged sub-segment falls into one of three categories:

1. **Numerical noise near zero** — filtered by the `MOR_CONV_THRESHOLD_CMS_YR` / `SZ_DIV_THRESHOLD_CMS_YR` USER CONFIG knobs (default ±0.5 cm/yr). No further action needed.
2. **MOR / transform label swaps** — the shared boundary section carries a single `gpml_mid_ocean_ridge` or `gpml_transform` label, but PTT and pyGPlates' local ridge-vs-transform inference isn't foolproof. At ridge-transform intersections along the Pacific paleo-ridge system in particular, transform arcs can inherit an MOR label (or vice versa), producing a legitimate wrong-sign flag on a sub-segment that isn't actually a MOR. These are **known false positives** — the algorithm is doing its arithmetic honestly, but the input label is wrong.
3. **Genuine model-level rotation-file inconsistencies** — cases where Z22's rotations actually predict wrong-sign motion at a genuinely-drawn MOR or SZ arc, because the two adjacent plates' rotations were derived from separate constraints that don't quite reconcile at their shared boundary. **These are the real bugs T17 is designed to expose.**

Categories 2 and 3 look identical on the map — a cyan segment is a cyan segment. Sorting them requires opening GPlates and inspecting each flagged location manually. Treat the maps as a first-pass triage that concentrates your attention on the \~1% of the plate boundary network worth looking at, not as a verdict on which sub-segments are 'wrong'.

**Audience**: intermediate → researcher (basic pyGPlates familiarity assumed).
**Difficulty**: ★★★.
**Runtime**: \~1 min for the 5 static snapshots; \~5-10 min for the 1-Myr video (101 frames).


## Data availability

**Plate model** — Zahirovic 2022 (`Zahirovic2022`), fetched via the Plate Model Manager on first run. This is the suite default. Any PMM model can be swapped in via `MODEL_NAME` in the USER CONFIGURATION block.

**Helper module** — `Notebooks/plate_model_debug/` (bundled with the suite). Wraps upstream code from the bundled `plate_model_debug` helper package (Ben Sculley + John Cannon, EarthByte), being made public via this tutorial suite.

**No CSV / NetCDF inputs required** — everything derives from the plate model's rotation file + topology GPMLs.

## Sources

- Sculley, B. & Cannon, J. (2025) *plate-model-debug*. EarthByte, University of Sydney. Companion code for these tutorials.
- Zahirovic, S., Eleish, A., Doss, S., Pall, J., Cannon, J., Pistone, M., Tetley, M.G., Young, A. & Fox, P. (2022). Subduction and carbonate platform interactions. *Geoscience Data Journal* 9(2), 371-383. https://doi.org/10.1002/gdj3.146


## Environment + imports


In [1]:
from pathlib import Path
import os, sys, warnings, tempfile, shutil
if Path("../data").exists() and not Path("data").exists():
    os.chdir("..")

import numpy as np
import gplately
import pygmt
import pygplates
from plate_model_manager import PlateModelManager

# Bundled helper module
sys.path.insert(0, str(Path("Notebooks").resolve()))
from plate_model_debug import diagnose_topology_convergence, calculate_plate_motion_arrows

print("Environment")
print(f"  python      {sys.version.split()[0]}")
for _m in (np, gplately, pygmt, pygplates):
    print(f"  {_m.__name__:11s} {getattr(_m, '__version__', 'n/a')}")


Environment
  python      3.12.5
  numpy       2.3.2
  gplately    2.0.0.post19+git.2cce7bb3
  pygmt       v0.18.0
  pygplates   1.0.0


In [2]:
# === USER CONFIGURATION =====================================================
# Plate model + anchor (Z22 is the tutorial-suite default).
MODEL_NAME             = "Zahirovic2022"
ANCHOR_PLATE_ID        = 0

# Static snapshot ages for the §3 pyGMT panels.
SNAPSHOT_AGES_MA       = [100, 125, 150, 175, 200]

# Video: 1-Myr cadence over 100-200 Ma (compact, ~5 MB).
VIDEO_START_MA         = 200        # oldest frame
VIDEO_END_MA           = 100        # youngest frame
VIDEO_CADENCE_MA       = 1          # frame every 1 Myr
VIDEO_FPS              = 10         # playback speed
VIDEO_WIDTH_PX         = 800        # frame width in pixels (compact)
VIDEO_CRF              = 28         # h264 quality (23=default, 28=compact)

# Convergence-anomaly thresholds (cm/yr). Non-zero values filter numerical
# noise at near-stationary boundaries and deforming-network edges. Ben's
# original NB uses 0.0 (everything above zero returned) but that produces
# many near-zero flags where the model's rotation is only marginally
# on the 'wrong' side of zero. A moderate threshold (0.5 cm/yr) keeps only
# ridges/trenches with genuinely significant wrong-sign convergence.
MOR_CONV_THRESHOLD_CMS_YR = 0.5    # MOR flagged if convergence velocity > this
SZ_DIV_THRESHOLD_CMS_YR   = -0.5   # SZ  flagged if convergence velocity < this

# Velocity delta-time and sampling for diagnose_topology_convergence
# (Ben's defaults; smaller sampling = denser, slower).
VELOCITY_DELTA_TIME_MYR = 1.0
SAMPLING_DISTANCE_KM    = 100.0     # ~ 100 km along-boundary spacing

# Map region + projection.
REGION_GLOBAL          = [-180, 180, -75, 80]
PROJECTION             = "N15c"     # Robinson, 15 cm wide

# Output dirs (gitignored per suite convention).
FRAMES_DIR             = Path("Notebooks/T59_debug_divergence_frames")
VIDEO_DIR              = Path("Notebooks/videos")
VIDEO_PATH             = VIDEO_DIR / "T59_debug_divergence.mp4"
# Plate-motion vector overlay.
ARROW_SPACING_DEG      = 15.0     # grid spacing for arrow sampling (global 15° = readable, regional 3-5°)
ARROW_VEL_SCALE        = 0.08     # cm map length per cm/yr (map-projection-dependent; tune to taste)
ARROW_MIN_SPEED_CM_YR  = 0.5      # arrows below this speed are suppressed (removes near-stationary noise)

# ============================================================================
print(f"  plate model:   {MODEL_NAME}")
print(f"  snapshots:     {SNAPSHOT_AGES_MA} Ma")
print(f"  video window:  {VIDEO_END_MA}-{VIDEO_START_MA} Ma at {VIDEO_CADENCE_MA}-Myr cadence")
print(f"  video output:  {VIDEO_PATH}  ({VIDEO_WIDTH_PX}px wide, crf {VIDEO_CRF})")


  plate model:   Zahirovic2022
  snapshots:     [100, 125, 150, 175, 200] Ma
  video window:  100-200 Ma at 1-Myr cadence
  video output:  Notebooks/videos/T59_debug_divergence.mp4  (800px wide, crf 28)


## 1. Load plate model + helper


In [3]:
pmm = PlateModelManager()
model = pmm.get_model(MODEL_NAME, data_dir="./gplately_data")

# PMM may return the rotation model as a list of paths; wrap if needed
_pmm_rot = model.get_rotation_model()
if not hasattr(_pmm_rot, "get_rotation"):
    rotation_model = pygplates.RotationModel(_pmm_rot)
else:
    rotation_model = _pmm_rot

# Load the topology features (feature-collection form, needed by resolve_topologies)
topology_features = [pygplates.FeatureCollection(f) for f in model.get_topologies()]

topological_model = pygplates.TopologicalModel(topology_features, rotation_model)

recon = gplately.PlateReconstruction(
    rotation_model=rotation_model,
    topology_features=model.get_topologies(),
    static_polygons=model.get_static_polygons(),
    anchor_plate_id=ANCHOR_PLATE_ID,
)
print(f"  plate model loaded: {MODEL_NAME}")
print(f"  topology feature collections: {len(topology_features)}")


  plate model loaded: Zahirovic2022
  topology feature collections: 4


## 2. Compute anomalies per snapshot

For each snapshot age, three calls to `diagnose_topology_convergence` — one filtered to MOR features (threshold: only return positive-convergence sub-segments = negative divergence), one filtered to SZ features (threshold: only return negative-convergence sub-segments = negative convergence), one filtered to transforms (no threshold — return all).


In [4]:
def diagnose_snapshot(time):
    """Return (mor_anomalies, sz_anomalies, transform_all) for one age snapshot."""
    # MORs — return sub-segments with POSITIVE convergence (i.e. wrong-sign)
    mor_features, mor_vels = diagnose_topology_convergence(
        rotation_model, topology_features, time,
        convergent_velocity_threshold_cms_yr=MOR_CONV_THRESHOLD_CMS_YR,
        boundary_feature_types=[pygplates.FeatureType.gpml_mid_ocean_ridge],
        velocity_delta_time=VELOCITY_DELTA_TIME_MYR,
        threshold_sampling_distance_radians=SAMPLING_DISTANCE_KM / pygplates.Earth.mean_radius_in_kms,
        anchor_plate_id=ANCHOR_PLATE_ID,
    )
    # SZs — return sub-segments with NEGATIVE convergence (i.e. wrong-sign)
    sz_features, sz_vels = diagnose_topology_convergence(
        rotation_model, topology_features, time,
        divergent_velocity_threshold_cms_yr=SZ_DIV_THRESHOLD_CMS_YR,
        boundary_feature_types=[pygplates.FeatureType.gpml_subduction_zone],
        velocity_delta_time=VELOCITY_DELTA_TIME_MYR,
        threshold_sampling_distance_radians=SAMPLING_DISTANCE_KM / pygplates.Earth.mean_radius_in_kms,
        anchor_plate_id=ANCHOR_PLATE_ID,
    )
    # Transforms — return everything (no threshold)
    tr_features, tr_vels = diagnose_topology_convergence(
        rotation_model, topology_features, time,
        boundary_feature_types=[pygplates.FeatureType.gpml_transform],
        velocity_delta_time=VELOCITY_DELTA_TIME_MYR,
        threshold_sampling_distance_radians=SAMPLING_DISTANCE_KM / pygplates.Earth.mean_radius_in_kms,
        anchor_plate_id=ANCHOR_PLATE_ID,
    )
    return (mor_features, mor_vels), (sz_features, sz_vels), (tr_features, tr_vels)

# Quick sanity call at 150 Ma
_mor, _sz, _tr = diagnose_snapshot(150.0)
print(f"  150 Ma:  {len(_mor[0])} MOR anomalies, {len(_sz[0])} SZ anomalies, "
      f"{len(_tr[0])} transform sub-segments (all shown)")


  150 Ma:  10 MOR anomalies, 6 SZ anomalies, 18 transform sub-segments (all shown)


## 3. Static snapshots at 100, 125, 150, 175, 200 Ma

One pyGMT map per snapshot. Colour coding:

- **Cyan lines** — MOR sub-segments with negative divergence (topology or rotation problem)
- **Magenta lines** — SZ sub-segments with negative convergence
- **Yellow lines** — transform sub-segments with any non-zero orthogonal velocity (colour intensity = magnitude)

Base map: coastlines + all plate-boundary topological sections (the continuous-backbone pattern) in grey, so the anomalous sub-segments stand out on top.


In [ ]:
def render_snapshot_map(time, out_path=None, width_cm=15):
    """Render one snapshot; write to `out_path` if given, else return the figure."""
    (mor_feats, _), (sz_feats, _), (tr_feats, tr_vels) = diagnose_snapshot(time)

    gplot = gplately.PlotTopologies(
        plate_reconstruction=recon,
        coastlines=model.get_coastlines(),
        continents=model.get_continental_polygons(),
        COBs=model.get_COBs(),
        time=float(time),
        plot_engine=gplately.PygmtPlotEngine(),
    )

    fig = pygmt.Figure()
    fig.basemap(region=REGION_GLOBAL, projection=f"N{width_cm}c", frame="af")

    # Continents in light grey (house style)
    try:
        gplot.plot_continents(fig, fill="gray95", pen="0.2p,gray40")
        gplot.plot_coastlines(fig, pen="0.3p,gray20")
    except Exception:
        pass

    # Continuous backbone of all topological sections in mid-grey
    try:
        engine = gplot._plot_engine if hasattr(gplot, "_plot_engine") else gplately.PygmtPlotEngine()
        engine.plot_geo_data_frame(fig, gplot.get_all_topological_sections(),
                                    pen="0.4p,gray60")
    except Exception:
        pass

    # --- Plate-motion vector overlay (dark-grey arrows under the anomaly layer)
    try:
        _snap = topological_model.topological_snapshot(float(time))
        _alons, _alats, _ae, _an = calculate_plate_motion_arrows(
            _snap, region=REGION_GLOBAL, spacing_deg=ARROW_SPACING_DEG)
        if len(_alons):
            _speed = np.sqrt(_ae ** 2 + _an ** 2)
            _azimuth = np.rad2deg(np.arctan2(_ae, _an))
            _m = _speed >= ARROW_MIN_SPEED_CM_YR
            if _m.any():
                fig.plot(x=_alons[_m], y=_alats[_m],
                          style="V0.15c+e+a35",
                          direction=[_azimuth[_m], ARROW_VEL_SCALE * _speed[_m]],
                          fill="gray25", pen="0.3p,gray25")
    except Exception as _e:
        print(f"    (skip velocity arrows: {type(_e).__name__}: {_e})")


    # Anomalous lines — extract polyline geometries and draw them
    def _plot_features(features, pen):
        for f in features:
            for geom in f.get_geometries():
                pts = geom.to_lat_lon_array()
                fig.plot(x=pts[:, 1], y=pts[:, 0], pen=pen)

    _plot_features(mor_feats, pen="1.6p,cyan")
    _plot_features(sz_feats,  pen="1.6p,magenta")
    _plot_features(tr_feats,  pen="0.8p,yellow2")

    fig.text(text=f"{int(time)} Ma  ({MODEL_NAME})  "
                   f"MOR anomalies: {len(mor_feats)}  |  "
                   f"SZ anomalies: {len(sz_feats)}  |  "
                   f"transform sub-segments: {len(tr_feats)}",
              position="TL", offset="0.25c/-0.25c", justify="TL",
              font="10p,Helvetica-Bold,black", fill="white", pen="0.5p,gray40")

    if out_path is not None:
        fig.savefig(str(out_path), dpi=100)
        return None
    return fig

# Render all 5 static snapshots
for _age in SNAPSHOT_AGES_MA:
    print(f"  rendering {_age} Ma snapshot ...")
    _f = render_snapshot_map(_age, width_cm=15)
    _f.show(width=900)


### How to read these maps

- **Cyan segments** on any ridge line = the two plates on either side of that segment are moving TOWARD each other in this model. Either the ridge geometry is drawn incorrectly (should be a trench or a transform), or the rotation model has the wrong sense of motion for the plates involved, or the plate-ID assignment is wrong.
- **Magenta segments** on any trench line = the two plates on either side are moving APART. Same three possible causes, but flipped.
- **Yellow segments** on any transform = the transform has a non-zero across-boundary velocity component. In a well-drawn model, transforms are pure strike-slip (velocity along the boundary, zero across it). Non-zero across-boundary velocity means the "transform" is functioning partly as a ridge or a trench — usually a legacy of an active-margin transition that was drawn as a transform but hasn't yet been re-typed.

**Threshold note.** By default `MOR_CONV_THRESHOLD_CMS_YR = 0.5` and `SZ_DIV_THRESHOLD_CMS_YR = -0.5` — a sub-segment is only flagged if its wrong-sign convergence exceeds 0.5 cm/yr in magnitude. Setting these to 0 (Ben's original default) flags many near-zero cases where the algorithm picks up numerical noise from stage rotations, ambiguous plate-ID assignments at deforming-network edges (Z22 has wide deforming networks in the Pacific), or sub-segments that inherit the MOR/SZ label from a shared boundary section but locally function as transforms at ridge-transform triple junctions. Raising the threshold isolates the sub-segments that are genuinely wrong-sign, not just marginally so.

**Known limitation: MOR / transform label swaps.** The feature-type of each sub-segment is inherited from the SHARED BOUNDARY SECTION it belongs to — one shared boundary section may span an entire ridge system that contains both true MOR arcs AND transform offsets, but the whole section carries a single `gpml_mid_ocean_ridge` label. At each ridge-transform intersection the transition is not modelled sub-segment-by-sub-segment; PTT and pyGPlates infer it from the local geometry, and that inference is not foolproof. Whenever a transform arc gets labelled MOR (or a MOR arc gets labelled transform) the algorithm computes the "wrong-sign" convergence velocity honestly, but the flag is a false positive because the sub-segment isn't actually a MOR at all. This is why you may see cyan clusters along the Pacific-Kula, Pacific-Farallon, and Pacific-Antarctic ridge systems where PTT's classifier and Z22's shared-boundary-section labelling disagree.

**Not all remaining flags are label swaps.** Even after removing near-zero cases (via the threshold above) and after acknowledging the label-swap false positives, a residual number of flagged sub-segments correspond to genuine model-level inconsistencies — cases where Z22's rotation file actually predicts wrong-sign motion at a genuinely-drawn MOR arc, because the two adjacent plates' rotations were derived from separate constraints that don't quite reconcile at their shared boundary. These are the real bugs T17 is designed to expose. Distinguishing them from label swaps requires opening GPlates and inspecting the boundary geometry manually at each flagged location — the maps below are a triage tool, not an authoritative catalogue of "true" errors.

**Watch for**

- **Snapshot 200 Ma** — early Mesozoic Panthalassa, lots of subduction, likely to show the most SZ anomalies where absolute-motion reconstructions of the Panthalassic plate are least constrained.
- **Snapshot 100 Ma** — mid-Cretaceous, well-constrained by preserved seafloor. Anomaly count should be at its lowest here.
- **Anomaly location, not just count** — a single cyan segment at the North Atlantic MOR is a different fix than a cluster of cyan segments along the entire Pacific-Farallon boundary.


## 4. Compact MP4 video at 1-Myr cadence over 100-200 Ma

Same rendering as §3, but for every 1-Myr step in the window. Frames are stitched with ffmpeg (via `imageio-ffmpeg`, which ships with a bundled binary — no external install needed). Video is written to `Notebooks/videos/T59_debug_divergence.mp4` (gitignored) and displayed inline via HTML5.

Runtime: \~5-10 minutes on a laptop for 101 frames at compact settings.

Same three-category interpretation as §3 applies to the video: flagged sub-segments are a mix of label swaps (category 2) and genuine rotation-file inconsistencies (category 3), and only manual GPlates inspection can sort them. Watching the flags come and go across the 100-200 Ma window is a useful signal though — sub-segments that appear for only a few frames as the topology changes are usually transient false positives at a moving triple junction; sub-segments that persist across many 1-Myr steps are more likely to be genuine model bugs.


In [6]:
# Build MP4 by rendering per-frame PNGs and stitching with imageio-ffmpeg.

FRAMES_DIR.mkdir(exist_ok=True, parents=True)
VIDEO_DIR.mkdir(exist_ok=True, parents=True)

# Frame width in cm (pygmt) computed from target pixel width at ~100 dpi
_target_dpi = 100
_frame_width_cm = VIDEO_WIDTH_PX / _target_dpi * 2.54

ages = np.arange(VIDEO_END_MA, VIDEO_START_MA + 1e-6, VIDEO_CADENCE_MA)
ages_desc = ages[::-1]  # oldest first for a "watch time run forward" playback

print(f"  will render {len(ages_desc)} frames at {_frame_width_cm:.1f} cm wide "
      f"(~{VIDEO_WIDTH_PX} px)")
print(f"  frame directory: {FRAMES_DIR}")

# Render frames (skip if already on disk — makes reruns cheap)
frame_paths = []
for i, _age in enumerate(ages_desc):
    _p = FRAMES_DIR / f"frame_{int(round(_age)):04d}Ma.png"
    if not _p.exists():
        render_snapshot_map(float(_age), out_path=_p, width_cm=_frame_width_cm)
    if i % 10 == 0:
        print(f"    frame {i+1}/{len(ages_desc)}  ({int(round(_age))} Ma)")
    frame_paths.append(_p)
print(f"  ✓ {len(frame_paths)} frames on disk")

# Encode with imageio-ffmpeg
import imageio.v2 as imageio
import imageio_ffmpeg

writer = imageio.get_writer(
    str(VIDEO_PATH), fps=VIDEO_FPS, codec="libx264",
    quality=None, ffmpeg_params=["-crf", str(VIDEO_CRF),
                                    "-preset", "veryfast",
                                    "-pix_fmt", "yuv420p"],
)
for _p in frame_paths:
    writer.append_data(imageio.imread(str(_p)))
writer.close()
print(f"  ✓ wrote {VIDEO_PATH}  ({VIDEO_PATH.stat().st_size / 1e6:.1f} MB)")

# Display inline
from IPython.display import Video
Video(str(VIDEO_PATH), embed=True, width=VIDEO_WIDTH_PX)


  will render 101 frames at 20.3 cm wide (~800 px)
  frame directory: Notebooks/T59_debug_divergence_frames
    frame 1/101  (200 Ma)
    frame 11/101  (190 Ma)
    frame 21/101  (180 Ma)
    frame 31/101  (170 Ma)
    frame 41/101  (160 Ma)
    frame 51/101  (150 Ma)
    frame 61/101  (140 Ma)
    frame 71/101  (130 Ma)
    frame 81/101  (120 Ma)
    frame 91/101  (110 Ma)
    frame 101/101  (100 Ma)
  ✓ 101 frames on disk
2026-07-21 14:23:36 - imageio_ffmpeg - WARNING - IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (826, 396) to (832, 400) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


[vost#0:0/libx264 @ 0x143704ac0] Multiple -pix_fmt options specified for stream 0, only the last option '-pix_fmt yuv420p' will be used.


  ✓ wrote Notebooks/videos/T59_debug_divergence.mp4  (0.9 MB)


## Extend this

- **Different plate model** — swap `MODEL_NAME` to `"Muller2019"`, `"Muller2022"`, `"Cao2024"`, `"Merdith2021"`, or any PMM model listed at <https://repo.gplates.org/webdav/pmm/config/models_v2.json>. Each will have a different anomaly pattern — comparing them side-by-side is a fast way to see which regions are well-constrained across models vs disputed.
- **Different time window** — bump `VIDEO_START_MA` / `VIDEO_END_MA` to cover 0-1000 Ma. Runtime scales linearly with frames; a 1000-frame video takes \~1 hour at these compact settings.
- **Your own plate model** — point `pygplates.RotationModel` at your rotation file and `pygplates.FeatureCollection` at your topology GPMLs, skip the `plate_model_manager` fetch, and re-run everything downstream. The helper module makes no assumption about which model you're using.
- **Tighter velocity thresholds** — set `convergent_velocity_threshold_cms_yr=0.5` (say) on the MOR call to only flag sub-segments with STRONGLY wrong-sign convergence (> 0.5 cm/yr toward each other). Filters out numerical noise near stationary plates.
- **Zoom into a specific region** — swap `REGION_GLOBAL` for a regional bounding box (e.g. `[-180, -60, -60, 15]` for the SE Pacific + South America) and change projection to Mercator (`M15c`) for high-resolution local diagnostics.

## Related resources

- Companion notebooks in this cluster: **T18** velocity magnitude at MORs, **T19** topology construction anomalies (gaps/overlaps + non-unique sections + missing polarity), **T20** feature extractability at subduction zones.

## References

- Zahirovic, S., Eleish, A., Doss, S., Pall, J., Cannon, J., Pistone, M., Tetley, M.G., Young, A. & Fox, P. (2022). Subduction and carbonate platform interactions. *Geoscience Data Journal* 9(2), 371-383. https://doi.org/10.1002/gdj3.146
- Müller, R.D., et al. (2019) A global plate model including lithospheric deformation along major rifts and orogens since the Triassic. *Tectonics* 38(6), 1884-1907. — alternative plate model.
- Mather, B.R., et al. (2024) Deep time spatio-temporal data analysis using pyGPlates with PlateTectonicTools and GPlately. *Applied Computing and Geosciences* 22, 100152.
